# File.cypher generator

### Imports

In [1]:
import pandas as pd

### Load the CSV files
If using a different files, change the value of the "pokemon_csv_path" and "player_csv_path" variables

In [ ]:
# Carica i file CSV
pokemon_csv_path = 'pokemon_data.csv'
player_csv_path = 'players.csv'
pokemon_data = pd.read_csv(pokemon_csv_path)
player_data = pd.read_csv(player_csv_path)

In [ ]:
# Genera lo script Cypher
cypher_script = []

# Creation of Type nodes
Using from pokemon_csv_path

In [ ]:
# Creazione di nodi Type
types = pd.concat([pokemon_data['type1'], pokemon_data['type2']]).dropna().drop_duplicates()
for type_name in types:
    type_name = type_name.replace("'", "\\'")  # Escaping
    cypher_script.append(f"MERGE (:Type {{type_name: '{type_name}'}});")

### Creation of Pokemon nodes and relations with Type nodes
Using data from pokemon_csv_path.
After a Pokémon node is created, it is linked to a Type node.


In [ ]:
# Creazione di nodi Pokemon
pokemons = pokemon_data[['name', 'type1', 'type2', 'hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']].drop_duplicates().fillna('')
for _, pokemon in pokemons.iterrows():
    pokemon_name = pokemon['name'].replace("'", "\\'").lower()
    cypher_script.append(f"""
    MERGE (p:Pokemon {{
        pokemon_name: '{pokemon_name}',
        hp: {pokemon['hp']}, attack: {pokemon['attack']}, defense: {pokemon['defense']},
        special_attack: {pokemon['special_attack']}, special_defense: {pokemon['special_defense']}, speed: {pokemon['speed']}
    }});
    """)
    
    # Relazioni con Type
    for type_name in [pokemon['type1'], pokemon['type2']]:
        if type_name:
            type_name = type_name.replace("'", "\\'")
            cypher_script.append(f"""
            MATCH (p:Pokemon {{pokemon_name: '{pokemon_name}'}}),
                  (t:Type {{type_name: '{type_name}'}})
            MERGE (p)-[:HAS_TYPE]->(t);
            """)

### Creation of Team nodes
Using data from player_csv_path.


In [ ]:
# Creazione di nodi Team e Nature
teams = player_data[['player_name', 'rank']].drop_duplicates()
for _, team in teams.iterrows():
    player_name = team['player_name'].replace("'", "\\'")
    cypher_script.append(f"MERGE (:Team {{player_name: '{player_name}', rank: {team['rank']}}});")

### Creation of Nature nodes
Using data from player_csv_path.

In [ ]:
natures = player_data[['nature']].drop_duplicates()
for _, nature in natures.iterrows():
    nature_name = nature['nature'].replace("'", "\\'")
    cypher_script.append(f"MERGE (:Nature {{nature_name: '{nature_name}'}});")

### Creation of relations between: Team-Pokemon, Pokemon-Type e Pokemon-Nature
Nodes created before are being linked following the data from player_csv_path. 

In [ ]:
# Relazioni Team-Pokemon, Pokemon-Type e Pokemon-Nature
for _, row in player_data.iterrows():
    player_name = row['player_name'].replace("'", "\\'")
    pokemon_name = row['pokemon_name'].replace("'", "\\'").lower()
    teratype = row['teratypes'].replace("'", "\\'").lower()
    nature_name = row['nature'].replace("'", "\\'")

    # Relazione Team-Pokemon
    cypher_script.append(f"""
    MATCH (t:Team {{player_name: '{player_name}'}}),
          (p:Pokemon {{pokemon_name: '{pokemon_name}'}})
    MERGE (t)-[:USE]->(p);
    """)

    # Relazione Pokemon-Type (teratype)
    cypher_script.append(f"""
    MATCH (p:Pokemon {{pokemon_name: '{pokemon_name}'}}),
          (t:Type {{type_name: '{teratype}'}})
    MERGE (p)-[:HAS_TERATYPE {{player_name: '{player_name}'}}]->(t);
    """)

    # Relazione Pokemon-Nature
    cypher_script.append(f"""
    MATCH (p:Pokemon {{pokemon_name: '{pokemon_name}'}}),
          (n:Nature {{nature_name: '{nature_name}'}})
    MERGE (p)-[:HAS_NATURE {{player_name: '{player_name}'}}]->(n);
    """)

### Saving the Cypher script to a file.cypher

In [ ]:
# Salva il Cypher script in un file
with open("neo4j_import.cypher", "w", encoding="utf-8") as file:
    file.write("\n".join(cypher_script))